In [17]:
import os
import os.path as op
import numpy as np
import nibabel as nib
import pandas as pd
from nilearn.maskers import NiftiMasker
from nilearn import datasets
from nilearn.image import threshold_img

## Define Dice Coefficient Function
The Dice coefficient measures the overlap between two binary masks, ranging from 0 (no overlap) to 1 (perfect overlap).

In [18]:
def dice_coefficient(mask1, mask2, brain_mask=None):
    """
    Calculate Dice similarity coefficient between two binary masks.
    
    Dice = 2 * |X ∩ Y| / (|X| + |Y|)
    
    Parameters:
    -----------
    mask1, mask2 : array-like
        Binary masks (1D or 3D arrays)
    brain_mask : array-like, optional
        Binary mask to restrict analysis to brain voxels
    
    Returns:
    --------
    dice : float
        Dice coefficient (0-1)
    """
    # Flatten arrays if needed
    mask1_flat = mask1.flatten()
    mask2_flat = mask2.flatten()
    
    # Apply brain mask if provided
    if brain_mask is not None:
        brain_mask_flat = brain_mask.flatten().astype(bool)
        mask1_flat = mask1_flat[brain_mask_flat]
        mask2_flat = mask2_flat[brain_mask_flat]
    
    # Calculate intersection and sizes
    intersection = np.sum(mask1_flat * mask2_flat)
    size1 = np.sum(mask1_flat)
    size2 = np.sum(mask2_flat)
    
    # Avoid division by zero
    if size1 + size2 == 0:
        return 0.0
    
    dice = (2.0 * intersection) / (size1 + size2)
    return dice

In [19]:
def weighted_dice_coefficient(map1, map2, brain_mask=None):
    """
    Calculate weighted Dice similarity coefficient between two continuous-valued maps.
    Weights voxels by their activation strength (absolute value).
    
    Weighted Dice = 2 * Σ(min(|X|, |Y|)) / (Σ|X| + Σ|Y|)
    
    Parameters:
    -----------
    map1, map2 : array-like
        Continuous-valued activation maps (1D or 3D arrays)
    brain_mask : array-like, optional
        Binary mask to restrict analysis to brain voxels
    
    Returns:
    --------
    weighted_dice : float
        Weighted Dice coefficient (0-1)
    """
    # Flatten arrays if needed
    map1_flat = map1.flatten()
    map2_flat = map2.flatten()
    
    # Apply brain mask if provided
    if brain_mask is not None:
        brain_mask_flat = brain_mask.flatten().astype(bool)
        map1_flat = map1_flat[brain_mask_flat]
        map2_flat = map2_flat[brain_mask_flat]
    
    # Use absolute values to handle negative activations
    map1_abs = np.abs(map1_flat)
    map2_abs = np.abs(map2_flat)
    
    # Calculate weighted intersection (minimum activation at each voxel)
    weighted_intersection = np.sum(np.minimum(map1_abs, map2_abs))
    
    # Calculate sum of activations
    sum1 = np.sum(map1_abs)
    sum2 = np.sum(map2_abs)
    
    # Avoid division by zero
    if sum1 + sum2 == 0:
        return 0.0
    
    weighted_dice = (2.0 * weighted_intersection) / (sum1 + sum2)
    return weighted_dice

In [20]:
def permutation_test_dice(map1, map2, dice_func, n_permutations=10000, seed=42):
    """
    Perform permutation test to assess statistical significance of Dice coefficient.
    
    Parameters:
    -----------
    map1, map2 : array-like
        Maps to compare (binary or continuous)
    dice_func : function
        Function to calculate Dice (dice_coefficient or weighted_dice_coefficient)
    n_permutations : int
        Number of permutations to perform
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    p_value : float
        Two-tailed p-value
    null_distribution : array
        Dice coefficients from permuted data
    """
    np.random.seed(seed)
    
    # Calculate observed Dice
    observed_dice = dice_func(map1, map2)
    
    # Initialize null distribution
    null_dice = np.zeros(n_permutations)
    
    # Perform permutations
    for i in range(n_permutations):
        # Randomly permute map2
        map2_permuted = np.random.permutation(map2)
        # Calculate Dice for permuted data
        null_dice[i] = dice_func(map1, map2_permuted)
    
    # Calculate two-tailed p-value
    p_value = np.mean(null_dice >= observed_dice)
    
    return p_value, null_dice, observed_dice

## Setup Directories and Masker

In [21]:
data_dir = "./dset"

# Setup masker
mask_img = datasets.load_mni152_brain_mask(resolution=1)
masker = NiftiMasker(mask_img=mask_img)
masker = masker.fit()

# Load MNI152 template to use as brain mask
template_img = datasets.load_mni152_template(resolution=1)
template_data = template_img.get_fdata()
brain_mask = (template_data != 0).astype(int)

print(f"Brain mask shape: {brain_mask.shape}")
print(f"Brain voxels: {np.sum(brain_mask)}")

Brain mask shape: (197, 233, 189)
Brain voxels: 1886539


## Define File Paths

In [22]:
# Group directories
group_drawn_dir = op.join(data_dir, "group-drawn/habenula")
group_avg_dir = op.join(data_dir, "group-avg/habenula")

# File paths for group average (1-sample) maps
drawn_1s_fn = op.join(group_drawn_dir, "averaged", "sub-group_task-rest_desc-1SampletTest_thresh.nii.gz")
avg_1s_fn = op.join(group_avg_dir, "averaged", "sub-group_task-rest_desc-1SampletTest_thresh.nii.gz")

# File paths for group comparison (2-sample) maps
drawn_2s_fn = op.join(group_drawn_dir, "difference", "sub-group_task-rest_desc-2SampletTest_thresh.nii.gz")
avg_2s_fn = op.join(group_avg_dir, "difference", "sub-group_task-rest_desc-2SampletTest_thresh.nii.gz")

print("File paths defined:")
print(f"Drawn 1-sample: {drawn_1s_fn}")
print(f"Avg 1-sample: {avg_1s_fn}")
print(f"Drawn 2-sample: {drawn_2s_fn}")
print(f"Avg 2-sample: {avg_2s_fn}")

File paths defined:
Drawn 1-sample: ./dset/group-drawn/habenula/averaged/sub-group_task-rest_desc-1SampletTest_thresh.nii.gz
Avg 1-sample: ./dset/group-avg/habenula/averaged/sub-group_task-rest_desc-1SampletTest_thresh.nii.gz
Drawn 2-sample: ./dset/group-drawn/habenula/difference/sub-group_task-rest_desc-2SampletTest_thresh.nii.gz
Avg 2-sample: ./dset/group-avg/habenula/difference/sub-group_task-rest_desc-2SampletTest_thresh.nii.gz


## Calculate Dice Coefficients for Group Average Maps (1-Sample)

In [23]:
# Check if files exist
if not op.exists(drawn_1s_fn):
    print(f"ERROR: File not found: {drawn_1s_fn}")
elif not op.exists(avg_1s_fn):
    print(f"ERROR: File not found: {avg_1s_fn}")
else:
    # Load thresholded maps
    drawn_1s_img = nib.load(drawn_1s_fn)
    avg_1s_img = nib.load(avg_1s_fn)
    
    # Convert to arrays (masker already applies brain mask)
    drawn_1s_arr = masker.transform(drawn_1s_img)
    avg_1s_arr = masker.transform(avg_1s_img)
    
    # Create binary masks (any non-zero value = 1)
    drawn_1s_binary = (drawn_1s_arr != 0).astype(int)
    avg_1s_binary = (avg_1s_arr != 0).astype(int)
    
    # Calculate binary Dice coefficient (masker already restricts to brain voxels)
    dice_1s = dice_coefficient(drawn_1s_binary, avg_1s_binary)
    
    # Calculate weighted Dice coefficient (uses activation strength)
    weighted_dice_1s = weighted_dice_coefficient(drawn_1s_arr, avg_1s_arr)
    
    # Perform permutation tests
    print("\n=== Group Average (1-Sample) ===")
    print("Running permutation tests (10,000 permutations)...")
    p_binary_1s, null_binary_1s, obs_binary_1s = permutation_test_dice(
        drawn_1s_binary, avg_1s_binary, dice_coefficient, n_permutations=10000
    )
    p_weighted_1s, null_weighted_1s, obs_weighted_1s = permutation_test_dice(
        drawn_1s_arr, avg_1s_arr, weighted_dice_coefficient, n_permutations=10000
    )
    
    print(f"Binary Dice Coefficient: {dice_1s:.4f} (p = {p_binary_1s:.4f})")
    print(f"Weighted Dice Coefficient: {weighted_dice_1s:.4f} (p = {p_weighted_1s:.4f})")
    print(f"Hand-drawn map voxels: {np.sum(drawn_1s_binary)}")
    print(f"Average habenula map voxels: {np.sum(avg_1s_binary)}")
    print(f"Overlapping voxels: {np.sum(drawn_1s_binary * avg_1s_binary)}")

/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You mig


=== Group Average (1-Sample) ===
Running permutation tests (10,000 permutations)...
Binary Dice Coefficient: 1.0000 (p = 1.0000)
Weighted Dice Coefficient: 0.7498 (p = 1.0000)
Hand-drawn map voxels: 1882989
Average habenula map voxels: 1882989
Overlapping voxels: 1882989
Binary Dice Coefficient: 1.0000 (p = 1.0000)
Weighted Dice Coefficient: 0.7498 (p = 1.0000)
Hand-drawn map voxels: 1882989
Average habenula map voxels: 1882989
Overlapping voxels: 1882989


In [24]:
# Check if files exist
if not op.exists(drawn_2s_fn):
    print(f"ERROR: File not found: {drawn_2s_fn}")
elif not op.exists(avg_2s_fn):
    print(f"ERROR: File not found: {avg_2s_fn}")
else:
    # Load thresholded maps
    drawn_2s_img = nib.load(drawn_2s_fn)
    avg_2s_img = nib.load(avg_2s_fn)
    
    # Convert to arrays (masker already applies brain mask)
    drawn_2s_arr = masker.transform(drawn_2s_img)
    avg_2s_arr = masker.transform(avg_2s_img)
    
    # Create binary masks (any non-zero value = 1)
    drawn_2s_binary = (drawn_2s_arr != 0).astype(int)
    avg_2s_binary = (avg_2s_arr != 0).astype(int)
    
    # Calculate binary Dice coefficient (masker already restricts to brain voxels)
    dice_2s = dice_coefficient(drawn_2s_binary, avg_2s_binary)
    
    # Calculate weighted Dice coefficient (uses activation strength)
    weighted_dice_2s = weighted_dice_coefficient(drawn_2s_arr, avg_2s_arr)
    
    # Perform permutation tests
    print("\n=== Group Comparison (2-Sample) ===")
    print("Running permutation tests (10,000 permutations)...")
    p_binary_2s, null_binary_2s, obs_binary_2s = permutation_test_dice(
        drawn_2s_binary, avg_2s_binary, dice_coefficient, n_permutations=10000
    )
    p_weighted_2s, null_weighted_2s, obs_weighted_2s = permutation_test_dice(
        drawn_2s_arr, avg_2s_arr, weighted_dice_coefficient, n_permutations=10000
    )
    
    print(f"Binary Dice Coefficient: {dice_2s:.4f} (p = {p_binary_2s:.4f})")
    print(f"Weighted Dice Coefficient: {weighted_dice_2s:.4f} (p = {p_weighted_2s:.4f})")
    print(f"Hand-drawn map voxels: {np.sum(drawn_2s_binary)}")
    print(f"Average habenula map voxels: {np.sum(avg_2s_binary)}")
    print(f"Overlapping voxels: {np.sum(drawn_2s_binary * avg_2s_binary)}")

/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You mig


=== Group Comparison (2-Sample) ===
Running permutation tests (10,000 permutations)...
Binary Dice Coefficient: 0.5144 (p = 1.0000)
Weighted Dice Coefficient: 0.3361 (p = 1.0000)
Hand-drawn map voxels: 927424
Average habenula map voxels: 943232
Overlapping voxels: 481146
Binary Dice Coefficient: 0.5144 (p = 1.0000)
Weighted Dice Coefficient: 0.3361 (p = 1.0000)
Hand-drawn map voxels: 927424
Average habenula map voxels: 943232
Overlapping voxels: 481146


In [25]:
# Create summary dataframe
summary_data = {
    "Analysis Type": ["Group Average (1-Sample)", "Group Comparison (2-Sample)"],
    "Binary Dice": [dice_1s, dice_2s],
    "Binary p-value": [p_binary_1s, p_binary_2s],
    "Weighted Dice": [weighted_dice_1s, weighted_dice_2s],
    "Weighted p-value": [p_weighted_1s, p_weighted_2s],
    "Hand-Drawn Voxels": [np.sum(drawn_1s_binary), np.sum(drawn_2s_binary)],
    "Avg Habenula Voxels": [np.sum(avg_1s_binary), np.sum(avg_2s_binary)],
    "Overlapping Voxels": [
        np.sum(drawn_1s_binary * avg_1s_binary),
        np.sum(drawn_2s_binary * avg_2s_binary)
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n=== Dice Similarity Summary ===")
print(summary_df.to_string(index=False))


=== Dice Similarity Summary ===
              Analysis Type  Binary Dice  Binary p-value  Weighted Dice  Weighted p-value  Hand-Drawn Voxels  Avg Habenula Voxels  Overlapping Voxels
   Group Average (1-Sample)     1.000000             1.0       0.749832               1.0            1882989              1882989             1882989
Group Comparison (2-Sample)     0.514414             1.0       0.336127               1.0             927424               943232              481146


## Save Results

In [26]:
# Save summary to CSV
output_fn = op.join(data_dir, "dice_similarity_results.csv")
summary_df.to_csv(output_fn, index=False)
print(f"\nResults saved to: {output_fn}")


Results saved to: ./dset/dice_similarity_results.csv
